## Data Mining from CountryEconomy Website

This notebook scrapes the 2024 electricity generation table from CountryEconomy.

The dataset is used to measure renewable electricity infrastructure. The main variable is `renewable_installed_capacity_share_pct`.

# Imports

In [19]:
import requests
import pandas as pd
from pathlib import Path
from io import StringIO


# Website Link

In [20]:
url = "https://countryeconomy.com/energy-and-environment/electricity-generation?year=2024"
output_file = Path("Datasets/countryeconomy_renewable_electricity_2024.csv")

url


'https://countryeconomy.com/energy-and-environment/electricity-generation?year=2024'

# Scrape Table from Website

In [21]:
print("Requesting website page...")
response = requests.get(url)

print("Status code:", response.status_code)

tables = pd.read_html(StringIO(response.text))
print("Tables found:", len(tables))

electricity_df = tables[0]
electricity_df.head()


Requesting website page...
Status code: 200
Tables found: 1


,Countries,Installed capacity MW,Generation GWh,Renewable installed capacity MW,Renewable generation GWh,Renewable percentage,Renewable percentage.1,Ch.
0,United States [+],1283644,4392552,451219,1068551,24.33%,NaN,1.34
1,United Kingdom [+],114640,267980,57947,140765,52.53%,NaN,2.24
2,Germany [+],287372,490177,177923,279331,56.99%,NaN,2.48
3,France [+],160877,537227,70084,150862,28.08%,NaN,-0.08
4,Japan [+],353288,907015,126313,227883,25.12%,NaN,-0.41


# Clean Scraped Data

In [22]:
electricity_df = electricity_df.copy()

# The table has one visual bar column, so only the useful data columns are kept.
electricity_df = electricity_df.iloc[:, [0, 1, 2, 3, 4, 5, 7]]

electricity_df.columns = [
    "country",
    "installed_capacity_mw",
    "generation_gwh",
    "renewable_installed_capacity_mw",
    "renewable_generation_gwh",
    "renewable_generation_share_pct",
    "renewable_generation_change_pct_points"
]

electricity_df.head()


,country,installed_capacity_mw,generation_gwh,renewable_installed_capacity_mw,renewable_generation_gwh,renewable_generation_share_pct,renewable_generation_change_pct_points
0,United States [+],1283644,4392552,451219,1068551,24.33%,1.34
1,United Kingdom [+],114640,267980,57947,140765,52.53%,2.24
2,Germany [+],287372,490177,177923,279331,56.99%,2.48
3,France [+],160877,537227,70084,150862,28.08%,-0.08
4,Japan [+],353288,907015,126313,227883,25.12%,-0.41


In [23]:
def clean_number(value):
    value = str(value)
    value = value.replace(",", "")
    value = value.replace("%", "")
    return pd.to_numeric(value, errors="coerce")

electricity_df["country"] = electricity_df["country"].str.replace("[+]", "", regex=False).str.strip()

number_columns = [
    "installed_capacity_mw",
    "generation_gwh",
    "renewable_installed_capacity_mw",
    "renewable_generation_gwh",
    "renewable_generation_share_pct",
    "renewable_generation_change_pct_points"
]

for column in number_columns:
    electricity_df[column] = electricity_df[column].apply(clean_number)

electricity_df["renewable_installed_capacity_share_pct"] = (
    electricity_df["renewable_installed_capacity_mw"] / electricity_df["installed_capacity_mw"] * 100
)

electricity_df["year"] = 2024
electricity_df["source_url"] = url

electricity_df.head()


,country,installed_capacity_mw,generation_gwh,renewable_installed_capacity_mw,renewable_generation_gwh,renewable_generation_share_pct,renewable_generation_change_pct_points,renewable_installed_capacity_share_pct,year,source_url
0,United States,1283644,4392552,451219,1068551,24.33,1.34,35.151413,2024,https://countryeconomy.com/energy-and-environm...
1,United Kingdom,114640,267980,57947,140765,52.53,2.24,50.546930,2024,https://countryeconomy.com/energy-and-environm...
2,Germany,287372,490177,177923,279331,56.99,2.48,61.913826,2024,https://countryeconomy.com/energy-and-environm...
3,France,160877,537227,70084,150862,28.08,-0.08,43.563716,2024,https://countryeconomy.com/energy-and-environm...
4,Japan,353288,907015,126313,227883,25.12,-0.41,35.753550,2024,https://countryeconomy.com/energy-and-environm...


# Save Dataset

In [24]:
final_columns = [
    "country",
    "year",
    "installed_capacity_mw",
    "generation_gwh",
    "renewable_installed_capacity_mw",
    "renewable_installed_capacity_share_pct",
    "renewable_generation_gwh",
    "renewable_generation_share_pct",
    "renewable_generation_change_pct_points",
    "source_url"
]

final_df = electricity_df[final_columns]

output_file.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(output_file, index=False)

print(f"Data for {len(final_df)} countries saved to '{output_file}'.")
final_df.head()


Data for 188 countries saved to 'Datasets\countryeconomy_renewable_electricity_2024.csv'.


,country,year,installed_capacity_mw,generation_gwh,renewable_installed_capacity_mw,renewable_installed_capacity_share_pct,renewable_generation_gwh,renewable_generation_share_pct,renewable_generation_change_pct_points,source_url
0,United States,2024,1283644,4392552,451219,35.151413,1068551,24.33,1.34,https://countryeconomy.com/energy-and-environm...
1,United Kingdom,2024,114640,267980,57947,50.546930,140765,52.53,2.24,https://countryeconomy.com/energy-and-environm...
2,Germany,2024,287372,490177,177923,61.913826,279331,56.99,2.48,https://countryeconomy.com/energy-and-environm...
3,France,2024,160877,537227,70084,43.563716,150862,28.08,-0.08,https://countryeconomy.com/energy-and-environm...
4,Japan,2024,353288,907015,126313,35.753550,227883,25.12,-0.41,https://countryeconomy.com/energy-and-environm...


# Why This Dataset Is Useful

The key indicator is `renewable_installed_capacity_share_pct`.

EVs can charge from any electricity source, but the EV transition is cleaner and stronger when a country has renewable electricity infrastructure. This variable shows how much of a country's electricity capacity is renewable in 2024.